In [ ]:
# retrieval_test.ipynb
import os, numpy as np, psycopg2
from pgvector.psycopg2 import register_vector
from sentence_transformers import SentenceTransformer

from src.

# 1) Connect (env already loaded)
dsn = os.getenv("SUPABASE_DB_URL")
assert dsn and "/postgres" in dsn, "Bad DSN"
if "sslmode=" not in dsn:
    dsn += ("&" if "?" in dsn else "?") + "sslmode=require"

conn = psycopg2.connect(dsn)
register_vector(conn)

# 2) Embedding model (384 dims)
model_name = "BAAI/bge-small-en-v1.5"
model = SentenceTransformer(model_name)

# Quick dim check
dim = model.get_sentence_embedding_dimension()
print("Embedding dim:", dim)
assert dim == 384, f"DB column must be vector({dim})"

# 3) Prepare query vector (normalized for cosine)
query = "What is prompt engineering?"
qvec = model.encode(query, normalize_embeddings=True).astype("float32").tolist()

# 4) Semantic search
SQL = """
SELECT 
  id,
  section_title,
  text,
  page_start,
  page_end,
  1 - (embedding <=> %s::vector) AS similarity
FROM chunks
ORDER BY embedding <=> %s::vector
LIMIT 5;
"""
with conn.cursor() as cur:
    cur.execute(SQL, (qvec, material_id, material_id, qvec))
    rows = cur.fetchall()

for r in rows:
    _id, title, txt, pstart, pend, sim = r
    print("\n" + "="*80)
    print(f"Section: {title}")
    if pstart is not None and pend is not None:
        print(f"Pages: {pstart}-{pend}")
    print(f"Similarity: {sim:.4f}")
    print("Text preview:", (txt or "")[:300].replace("\n"," ") + "...")

In [ ]:
import os, sys, psycopg2
dsn = os.getenv("SUPABASE_DB_URL")
if not dsn:
    raise RuntimeError("SUPABASE_DB_URL not set. Did you load .env / export it?")

if "/postgres" not in dsn:
    raise RuntimeError("Connection string must end with /postgres")

conn = psycopg2.connect(dsn)  # requires ?sslmode=require
with conn.cursor() as cur:
    cur.execute("select current_database(), current_user;")
    print(cur.fetchone())

In [ ]:
import os, sys, pathlib
print("CWD:", os.getcwd())
print("Kernel python:", sys.executable)
print("Env present here?:", pathlib.Path(".env").resolve(), pathlib.Path(".env").exists())

In [ ]:
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# Search upward from the current notebook folder and load the first `.env` it finds
env_path = find_dotenv(filename=".env", usecwd=True)
print("Loaded .env from:", env_path if env_path else "NOT FOUND")
if not env_path:
    # Fallback: compute repo root (4 levels up: notebooks -> test -> src -> repo-root)
    repo_root = Path.cwd().parents[3]
    env_path = repo_root / ".env"
    print("Fallback path:", env_path)
load_dotenv(env_path, override=False)

import os
dsn = os.getenv("SUPABASE_DB_URL")
assert dsn, "SUPABASE_DB_URL not set — check .env path/content"
print("OK: SUPABASE_DB_URL loaded")

In [ ]:
# retrieval_test_simple.ipynb
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import os
from supabase import create_client, Client

# ---------------------------------------------------------------------
# STEP 1 — Robust .env loader (your version)
# ---------------------------------------------------------------------
env_path = find_dotenv(filename=".env", usecwd=True)
print("Loaded .env from:", env_path if env_path else "NOT FOUND")
if not env_path:
    # Fallback: compute repo root (4 levels up: notebooks -> test -> src -> repo-root)
    repo_root = Path.cwd().parents[3]
    env_path = repo_root / ".env"
    print("Fallback path:", env_path)

load_dotenv(env_path, override=False)

# ---------------------------------------------------------------------
# STEP 2 — Retrieve keys from .env
# ---------------------------------------------------------------------
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError("Missing SUPABASE_URL or SUPABASE_SERVICE_ROLE_KEY in your .env file")

print("✅ SUPABASE_URL and SUPABASE_SERVICE_ROLE_KEY loaded")

# ---------------------------------------------------------------------
# STEP 3 — Create client
# ---------------------------------------------------------------------
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# ---------------------------------------------------------------------
# STEP 4 — Test fetch
# ---------------------------------------------------------------------
response = supabase.table("chunks").select("*").limit(5).execute()

if response.error:
    raise RuntimeError(f"❌ Supabase error: {response.error}")

data = response.data
print(f"✅ Retrieved {len(data)} rows")
for row in data:
    print("-" * 60)
    for k, v in row.items():
        print(f"{k}: {v}")